### RETINA AI HACKATHON — Complete Solution 

--import libraries--

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
import xgboost as xgb

print("✅ All imports done!")

✅ All imports done!


--load data--

In [2]:
train   = pd.read_csv('/kaggle/input/competitions/retina-ai-predict-student-dropout-risk-with-deep-learning/train.csv')
test    = pd.read_csv('/kaggle/input/competitions/retina-ai-predict-student-dropout-risk-with-deep-learning/test.csv')
attend  = pd.read_csv('/kaggle/input/competitions/retina-ai-predict-student-dropout-risk-with-deep-learning/Attendance_series.csv')
counsel = pd.read_csv('/kaggle/input/competitions/retina-ai-predict-student-dropout-risk-with-deep-learning/Counsellor_notes.csv')

print("✅ Data Loaded!")
print("Train shape  :", train.shape)
print("Test shape   :", test.shape)
print("Attendance   :", attend.shape)
print("Counsellor   :", counsel.shape)

✅ Data Loaded!
Train shape  : (12000, 18)
Test shape   : (3000, 17)
Attendance   : (1048575, 5)
Counsellor   : (15000, 2)


-- explore data --

In [3]:
print("=== TRAIN DATA ===")
print(train.head(3))
print()
print("=== TARGET (dropout_risk) ===")
print(train['dropout_risk'].value_counts())
print()
print("=== MISSING VALUES ===")
print(train.isnull().sum())

=== TRAIN DATA ===
  student_id branch gender hostel_status family_income parent_education  \
0   STU03539   AIML      M        Hostel          High        Bachelors   
1   STU12726     CE      M        Hostel        Medium          Masters   
2   STU01218   AIML      F   Day Scholar           Low      High School   

   scholarship  part_time_job  commute_time_mins  screen_time_hours  \
0            1              0                NaN           3.256760   
1            0              0               67.0           5.088147   
2            1              0                NaN           5.559880   

   cgpa_sem1  cgpa_sem2  cgpa_sem3  cgpa_sem4  backlogs_sem1  backlogs_sem2  \
0       7.20       7.50       7.54       7.89              0              0   
1       5.12       5.02       4.82       5.14              1              0   
2       7.92       8.86       9.26       9.00              0              1   

   backlogs_sem3  dropout_risk  
0              0             0  
1           

-- Attendance Features --****

In [4]:
# Subject wise average attendance
att_subject = (attend
    .groupby(['student_id', 'subject'])['attendance_pct']
    .mean().unstack(fill_value=0).reset_index())
att_subject.columns = ['student_id'] + [f'att_{c}' for c in att_subject.columns[1:]]

# Overall attendance stats
att_overall = (attend
    .groupby('student_id')['attendance_pct']
    .agg(att_mean='mean',
         att_min='min',
         att_std='std')
    .reset_index())

# Semester wise attendance
att_sem = (attend
    .groupby(['student_id', 'semester'])['attendance_pct']
    .mean().unstack(fill_value=0).reset_index())
att_sem.columns = ['student_id'] + [f'att_sem{int(c)}' for c in att_sem.columns[1:]]

# Merge all attendance features
att_features = att_subject.merge(att_overall, on='student_id').merge(att_sem, on='student_id')

print("✅ Attendance features created!")
print("Shape:", att_features.shape)
print(att_features.head(3))

✅ Attendance features created!
Shape: (15000, 10)
  student_id  att_Core_1  att_Core_2  att_Elective  att_mean  att_min  \
0   STU00001    0.791058    0.812209      0.817417  0.806669   0.1276   
1   STU00002    0.918333    0.942513      0.879283  0.913447   0.1816   
2   STU00003    0.729117    0.727174      0.773439  0.743041   0.1237   

    att_std  att_sem1  att_sem2  att_sem3  
0  0.214452  0.821704  0.791467  0.806850  
1  0.175171  0.959000  0.871404  0.909618  
2  0.209314  0.742754  0.751625  0.733991  


 NLP Features from Counsellor Notes
python ---

In [5]:
# Fill missing notes
counsel['counsellor_note'] = counsel['counsellor_note'].fillna('no notes')

# Risk keywords
high_risk_words   = ['dropout','failed','absent','depressed','demotivated',
                     'backlog','struggling','stress','quit','family issue']
medium_risk_words = ['concern','advised','improve','focus','tutoring',
                     'irregular','weak','moderate']
low_risk_words    = ['performing well','good','excellent','motivated',
                     'consistent','regular']

# Score each note
def risk_score(text):
    text = str(text).lower()
    h = sum(1 for w in high_risk_words   if w in text)
    m = sum(1 for w in medium_risk_words if w in text)
    l = sum(1 for w in low_risk_words    if w in text)
    return pd.Series({
        'nlp_high_risk' : h,
        'nlp_med_risk'  : m,
        'nlp_low_risk'  : l,
        'nlp_net_score' : l - h
    })

counsel_scores = counsel['counsellor_note'].apply(risk_score)
counsel_feat   = pd.concat([counsel[['student_id']], counsel_scores], axis=1)

# TF-IDF
tfidf = TfidfVectorizer(max_features=100, stop_words='english', ngram_range=(1,2))
tfidf.fit(counsel['counsellor_note'])

print("✅ NLP features created!")
print(counsel_feat.head(3))

✅ NLP features created!
  student_id  nlp_high_risk  nlp_med_risk  nlp_low_risk  nlp_net_score
0   STU00001              0             0             1              1
1   STU00002              0             2             0              0
2   STU00003              1             2             0             -1


-- Merge All Features --

In [6]:
def build_features(df):
    df = df.merge(att_features,  on='student_id', how='left')
    df = df.merge(counsel_feat,  on='student_id', how='left')
    return df

train = build_features(train)
test  = build_features(test)

print("✅ All features merged!")
print("Train shape:", train.shape)
print("Test shape :", test.shape)

✅ All features merged!
Train shape: (12000, 31)
Test shape : (3000, 30)


-- Feature Engineering --

In [7]:
for df in [train, test]:
    # CGPA features
    df['cgpa_avg']   = df[['cgpa_sem1','cgpa_sem2','cgpa_sem3','cgpa_sem4']].mean(axis=1)
    df['cgpa_min']   = df[['cgpa_sem1','cgpa_sem2','cgpa_sem3','cgpa_sem4']].min(axis=1)
    df['cgpa_trend'] = df['cgpa_sem4'] - df['cgpa_sem1']

    # Backlog features
    df['total_backlogs']    = df['backlogs_sem1'] + df['backlogs_sem2'] + df['backlogs_sem3']
    df['backlog_trend']     = df['backlogs_sem3'] - df['backlogs_sem1']

    # Risk flags
    df['low_cgpa_flag']       = (df['cgpa_avg'] < 5.5).astype(int)
    df['high_backlog_flag']   = (df['total_backlogs'] > 3).astype(int)

print("✅ Feature engineering done!")
print("New features added: cgpa_avg, cgpa_trend, total_backlogs, etc.")

✅ Feature engineering done!
New features added: cgpa_avg, cgpa_trend, total_backlogs, etc.


-- Encode Categorical Columns --

In [8]:
cat_cols = ['branch','gender','hostel_status','family_income','parent_education']

le = LabelEncoder()
for col in cat_cols:
    train[col] = train[col].fillna('Unknown')
    test[col]  = test[col].fillna('Unknown')
    combined   = pd.concat([train[col], test[col]])
    le.fit(combined)
    train[col] = le.transform(train[col])
    test[col]  = le.transform(test[col])

# Fill missing numeric values
train_num = train.select_dtypes(include=[np.number]).columns
test_num  = test.select_dtypes(include=[np.number]).columns
train[train_num] = train[train_num].fillna(train[train_num].median())
test[test_num]   = test[test_num].fillna(train[test_num].median())

print("✅ Encoding done!")

✅ Encoding done!


--  Prepare X and y --

In [9]:
drop_cols    = ['student_id', 'dropout_risk']
feature_cols = [c for c in train.columns if c not in drop_cols]

X_train = train[feature_cols]
y_train = train['dropout_risk']
X_test  = test[feature_cols]

print("✅ Data ready!")
print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"Total features: {X_train.shape[1]}")

✅ Data ready!
X_train shape : (12000, 36)
X_test shape  : (3000, 36)
Total features: 36


--  Train XGBoost Model --

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_model = xgb.XGBClassifier(
    n_estimators     = 500,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    eval_metric      = 'mlogloss',
    random_state     = 42,
    n_jobs           = -1
)

xgb_scores = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='accuracy')
print(f"✅ XGBoost CV Accuracy: {xgb_scores.mean():.4f} ± {xgb_scores.std():.4f}")

✅ XGBoost CV Accuracy: 0.6928 ± 0.0034


-- Train Random Forest Model --

In [11]:
rf_model = RandomForestClassifier(
    n_estimators = 300,
    max_depth    = 12,
    random_state = 42,
    n_jobs       = -1
)

rf_scores = cross_val_score(rf_model, X_train, y_train, cv=cv, scoring='accuracy')
print(f"✅ Random Forest CV Accuracy: {rf_scores.mean():.4f} ± {rf_scores.std():.4f}")

✅ Random Forest CV Accuracy: 0.6974 ± 0.0036


--  Final Training on Full Data --

In [12]:
print("Training XGBoost...")
xgb_model.fit(X_train, y_train)

print("Training Random Forest...")
rf_model.fit(X_train, y_train)

print("✅ Both models trained!")

Training XGBoost...
Training Random Forest...
✅ Both models trained!


--  Ensemble Predictions --

In [13]:
xgb_proba      = xgb_model.predict_proba(X_test)
rf_proba       = rf_model.predict_proba(X_test)

# XGBoost ko 60%, Random Forest ko 40% weight
ensemble_proba = 0.6 * xgb_proba + 0.4 * rf_proba
final_preds    = np.argmax(ensemble_proba, axis=1)

print("✅ Predictions done!")
print(pd.Series(final_preds).value_counts().sort_index())
print("\n0=Low Risk | 1=Medium Risk | 2=High Risk")

✅ Predictions done!
0    2211
1     452
2     337
Name: count, dtype: int64

0=Low Risk | 1=Medium Risk | 2=High Risk


-- Save Submission File --

In [14]:
submission = pd.DataFrame({
    'student_id'  : test['student_id'],
    'dropout_risk': final_preds
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

print("🎉 submission.csv saved!")
print(submission.head(10))

🎉 submission.csv saved!
  student_id  dropout_risk
0   STU03679             0
1   STU11070             0
2   STU13561             2
3   STU00061             0
4   STU02416             1
5   STU14493             0
6   STU14070             2
7   STU12038             0
8   STU13254             0
9   STU12397             0


In [15]:
import pandas as pd
sub = pd.read_csv('/kaggle/input/competitions/retina-ai-predict-student-dropout-risk-with-deep-learning/sample_submission.csv')
print(sub.shape)
print(sub.head(10))

(3000, 2)
  student_id  dropout_risk
0   STU03679             0
1   STU11070             0
2   STU13561             0
3   STU00061             0
4   STU02416             0
5   STU14493             0
6   STU14070             0
7   STU12038             0
8   STU13254             0
9   STU12397             0
